# Why we care

A lot of data is inherently location-based and this trend is increasing.  
- Any data that is county, state, country, city, etc. is tied to a geographic location on the ground
- Addresses
- Photos
- Businesses
- Satelite imagery

In order to unlock the full power of this data or to visualize it, we need to really treat it as geographic data.  

Sometimes it's important to do things like:
- Test if an address is within a county
- Aggregate data
- Figure out distances between points

**This lecture will cover:**
- using maps to display data
- other operations on geographic data

# A few mapping basics

**Projections:**  
There are a lot of different projections that are used.  A projection refers to the process used to convert points on the earth (more or less a sphere) to a flat surface like a map or computer monitor.

(This is largely beyond our scope here, but it is can be very important to pay attention to if one is going to do a lot with GIS data.)

**Latitude** is degrees north or south of the equator.  (-90 to 90) Lines of latitude are parallel.

**Longitude** is degrees east or west of the prime meridian (-180 to 180).  Lines of longitude converge at the poles (not parallel).

# Libraries for mapping in Python
There are several choices for libraries for creating maps.  A few of them include:
- matplotblib- basemap - [Geographic Data with Basemap - Python Data Science Handbook](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/04.13-Geographic-Data-With-Basemap.ipynb)
- Plotly Express - https://plotly.com/python/maps/

- [Altair](https://altair-viz.github.io/gallery/index.html#maps) - Another diverse plotting library that includes some mapping options
- [Folium](https://python-visualization.github.io/folium/) - Specifically for mapping- powerful, interactive maps based on leaflet.js
  

## References

- How to Lie with Maps
  - https://en.wikipedia.org/wiki/How_to_Lie_with_Maps
  - https://www.amazon.com/How-Lie-Maps-Mark-Monmonier/dp/0226534219
- QGIS
  - https://www.linkedin.com/learning/learning-qgis-2/
  - https://learning.oreilly.com/library/view/-/9781098104788/

In [ ]:
%%capture
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
%%capture
# Get updated version of plotly
!pip install plotly -U


In [ ]:
%%capture
!pip install geopandas


In [ ]:
import json
import numpy as np
import pandas as pd
import requests
import plotly.express as px
import geopandas as gpd


# COVID Data

**Choropleth** is really just a fancy name for a color-coded map where the color indicates something like population density or the number of positive covid cases.




Make request for the data.

In [ ]:
!curl -L -I https://covidtracking.com/data/download/all-states-history.csv


In [ ]:
!curl -L -O https://covidtracking.com/data/download/all-states-history.csv


In [ ]:
!head all-states-history.csv


In [ ]:
%%bash
# Verify that the downloaded data is the same data is in the /Data/ folder
# md5sum /content/drive/MyDrive/Data/all-states-history.csv
curl -L -s https://ddc-datascience.s3.amazonaws.com/all-states-history.csv | md5sum
md5sum all-states-history.csv


In [ ]:
# Pick one source and read it in.
# covid_csv="/content/drive/MyDrive/Data/all-states-history.csv"
covid_csv="all-states-history.csv"
covid_csv


Turn data into data frame and take a peek.

In [ ]:
state_df = pd.read_csv(covid_csv)
state_df.head()


See how many entries we have.

In [ ]:
state_df.shape


In [ ]:
state_df.info()

We have an entry for every day for every state and we just want the most recent data.

In [ ]:
type(state_df['date'][0])


Dates are currently strings.  We need to turn them into DateTime objects.

In [ ]:
state_df['date'] = pd.to_datetime(state_df['date'])
state_df.info()

In [ ]:
state_df['date'].sample(n=5, random_state=42)


Now we have it as a Timestamp

In [ ]:
state_df['date'].max()


Grab the latest data they do have which is March 7th, 2021.

In [ ]:
filter = ( state_df['date']==state_df['date'].max() )
march_df = state_df[ filter ]


In [ ]:
len(march_df)

In [ ]:
march_df.head()


In [ ]:
march_df.columns


## Choropleth Map

In [ ]:
fig = px.choropleth(march_df,
                    locations='state',
                    locationmode='USA-states',
                    color='positive',
                    color_continuous_scale="Viridis",
                    scope="usa",
                    labels={'positive':'Positive Tests'}
)
fig.show() ;


## Choropleth with Animation

Now let's look at April 2020.

In [ ]:
# Look at April 2020
filter = ( (state_df['date'] >= '2020-04-01') & (state_df['date'] < '2020-05-01') )
covid_apr = state_df[ filter ]
covid_apr_2 = covid_apr.copy()

# Create new column that is the day of the month
covid_apr_2['day'] = covid_apr['date'].dt.day

# Sort data frame by day
covid_apr_2.sort_values(by = 'day', inplace = True)


In [ ]:
covid_apr_2

In [ ]:
fig = px.choropleth(covid_apr_2,
                    locations='state',
                    locationmode='USA-states',
                    color='hospitalizedCurrently',
                    color_continuous_scale="Viridis",
                    scope="usa",
                    labels={'positive':'Positive Tests'},
                    animation_frame = 'day',
                    range_color=(0, 19000),
)
fig.show() ;


In [ ]:
# export animation to HTML file for viewing in an HTML page
fig.write_html(file="map.html")

# ABQ Police Incidents

- [APD ArcGIS REST service](https://coagisweb.cabq.gov/arcgis/rest/services/public/APD_Incidents/MapServer/0/query)
- [APD ArcGIS REST Docs](https://coagisweb.cabq.gov/arcgis/sdk/rest/index.html#//02ss0000000r000000)


Import data and take a look at it.

In [ ]:
query = 'https://coagisweb.cabq.gov/arcgis/rest/services/public/APD_Incidents/MapServer/0/query?'
attr = '''
  where=1%3D1
  text=
  objectIds=
  time=
  geometry=
  geometryType=esriGeometryEnvelope
  inSR=
  spatialRel=esriSpatialRelIntersects
  relationParam=
  outFields=*
  returnGeometry=true
  maxAllowableOffset=
  geometryPrecision=
  outSR=4326
  returnIdsOnly=false
  returnCountOnly=false
  orderByFields=
  groupByFieldsForStatistics=
  outStatistics=
  returnZ=false
  returnM=false
  gdbVersion=
  returnDistinctValues=false
  f=pjson
'''.split()
incidents_url = query + "&".join(attr)
incidents_url


Take a look at one entry

In [ ]:
# # Use curl to fetch the data as JSON
!curl -L "{incidents_url}" > apd.incidents.json


In [ ]:
!head apd.incidents.json


In [ ]:
# ... or download an older copy
!curl -L 'https://ddc-datascience.s3.amazonaws.com/apd.incidents.json' > apd.incidents.json

In [ ]:
ls -la apd*

In [ ]:
# Peak at the first 300 characters
!head -c 300 apd.incidents.json


In [ ]:
!jq . apd.incidents.json | head -200


In [ ]:
# Verify that the file downloaded is that same as what's in the data folder ( will likely be different with new data )
!ls -la /content/drive/MyDrive/Data/apd.incidents.json apd.incidents.json
!md5sum /content/drive/MyDrive/Data/apd.incidents.json apd.incidents.json


In [ ]:
# Extract the first feature using jq
!jq .features[0] apd.incidents.json


In [ ]:
# Load the data into a Python dictionary
with open("apd.incidents.json") as file:
  incidents_import = json.loads(file.read())


In [ ]:
# Look at the keys
incidents_import.keys()


In [ ]:
# Grab the first incident
# incidents_import = requests.get(incidents_url).json()  ## <-- this should have worked, but didn't
features = incidents_import['features']
first = features[0]
first


Turn our data into a dataframe and drop NAs.

In [ ]:
incidents = pd.DataFrame(features)
incidents.dropna(inplace=True)
incidents.head()


In [ ]:
incidents.info()


In [ ]:
type(incidents.attributes[0])


In [ ]:
type(incidents.geometry[0])


Extract data into separate columns.

In [ ]:
incidents['geometry'][:10]

In [ ]:
incidents['geometry'][:10].apply(lambda entry: entry['x'])

In [ ]:
incidents['longitude'] = incidents['geometry'].apply(lambda entry: entry['x'])
incidents['latitude'] = incidents['geometry'].apply(lambda entry: entry['y'])
incidents['type'] = incidents['attributes'].apply(lambda entry: entry['IncidentType'])
incidents['address'] = incidents['attributes'].apply(lambda entry: entry['BlockAddress'])
incidents['datetime'] = incidents['attributes'].apply(lambda entry: entry['ReportDateTime'])

incidents.head()


In [ ]:
incidents.info()

In [ ]:
incidents["datetime"].max()

In [ ]:
incidents["type"].value_counts()

In [ ]:
incidents["type"].value_counts().sum()

In [ ]:
# Take the first 500 incidents ( or last 500 )
incidents_500 = incidents.sort_values( by = "datetime" ).iloc[-500:]
incidents_500

## Scatter Mapbox

In [ ]:
px.scatter_mapbox?

'open-street-map'`,
    `'white-bg'`, `'carto-positron'`, `'carto-darkmatter'`, `'stamen-
    terrain'`, `'stamen-toner'`, `'stamen-watercolor'

In [ ]:
fig = px.scatter_map(incidents_500,
                        lat="latitude",
                        lon="longitude",
                        color = 'type',
                        color_discrete_sequence=px.colors.cyclical.IceFire,
                        zoom = 10,
                        # mapbox_style = 'stamen-watercolor',
                        # mapbox_style = 'carto-positron',
                        map_style = 'open-street-map',
                        )
fig.update_traces(marker={'size': 16})
fig.show()

## Heatmap

In [ ]:
# Just look at loud parties
parties = incidents[incidents['type'].str.contains('LOUD PARTY')].copy()
parties

In [ ]:
px.density_map?


In [ ]:
fig = px.density_map(parties,
                        lat="latitude",
                        lon="longitude",
                        zoom = 12,
                        # mapbox_style="stamen-terrain"
                        map_style = 'open-street-map',
                        )
fig.show()

## United States Census Tracts & Crime

References:
- [United States census - wikipedia]( https://en.wikipedia.org/wiki/United_States_census )
- [US Census American Community Survey - wikipedia]( https://en.wikipedia.org/wiki/American_Community_Survey )
- [US Census website]( https://www.census.gov/)
- [Standard Hierarchy of Census Geographic Entities]( https://www2.census.gov/geo/pdfs/reference/geodiagram.pdf )
- [US Census Data]( https://www.census.gov/data.html )
- [Census Data via FTP]( https://www.census.gov/programs-surveys/acs/data/data-via-ftp.html )
- [Census Data via API]( https://www.census.gov/programs-surveys/acs/data/data-via-api.html )




In [ ]:
from math import sqrt


In [ ]:
sqrt(2)


In [ ]:
import pandas as pd


Census tracts from: https://github.com/arcee123/GIS_GEOJSON_CENSUS_TRACTS

In [ ]:
tracts_url = 'https://raw.githubusercontent.com/arcee123/GIS_GEOJSON_CENSUS_TRACTS/master/35.geojson'
tracts = gpd.read_file(tracts_url)
tracts.head()


In [ ]:
tracts.info()

In [ ]:
tracts.plot() ;


In [ ]:
parties.head()

In [ ]:
parties.rename(columns={"geometry":"location"}, inplace=True)
parties.head()


In [ ]:
parties.info()

In [ ]:
# Create geometry for parties data
parties_geo = gpd.GeoDataFrame(
    parties,
    geometry=gpd.points_from_xy(
        parties.longitude,
        parties.latitude
    )
)
parties_geo.head()


In [ ]:
parties_geo.info()

See if a point is inside a tract polygon

In [ ]:
tracts.iloc[0]['geometry']


In [ ]:
tracts['geometry'].iloc[0]


In [ ]:
tracts['geometry'][0]


In [ ]:
tracts0 = tracts.geometry.iloc[0]
parties0 = parties_geo.geometry.iloc[0]


In [ ]:
tracts0


In [ ]:
parties0


In [ ]:
tracts0.contains(parties0)


In [ ]:
dict(enumerate(dir(tracts0)))


In [ ]:
getattr(tracts0, "exterior" )

In [ ]:
# all the methods/attributes
[ x for x in dir(tracts0) if '_' not in x ]


In [ ]:
type(tracts0)

In [ ]:
type(parties0)

Find the centroid

In [ ]:
print(tracts0.centroid)


Pull out lat or long for a point

In [ ]:
[ x for x in dir(parties0) if '_' not in x ]


In [ ]:
parties0.x


In [ ]:
parties0.y


In [ ]:
parties0.xy


Distance between points

In [ ]:
parties1 = parties_geo.geometry.iloc[1]


In [ ]:
import geopy.distance


In [ ]:
dist = geopy.distance.geodesic((parties0.y, parties0.x), (parties1.y, parties1.x))
dist.kilometers


In [ ]:
dist.miles


### Count the number of loud party incidents in each tract ... using a for-loop.

In [ ]:
parties

In [ ]:
zfoo = np.full(len(tracts), 0)

for idx, incident in parties_geo.iterrows():
  for idx, tract in tracts.iterrows():
    if tract['geometry'].contains(incident['geometry']):
      zfoo[idx] += 1
      break

tracts['incidents'] = zfoo


In [ ]:
tracts

In [ ]:
filter = tracts['incidents'] > 4
tracts[ filter ]


In [ ]:
parties_geo.shape, tracts.shape

### Using spatial join: sjoin


In [ ]:
gpd.sjoin?


In [ ]:
parties_geo.sindex.valid_query_predicates


In [ ]:
# create DF of tracts that have parties
joined_gpd = gpd.sjoin(
    left_df = parties_geo.set_crs( tracts.crs, allow_override = True ),
    right_df = tracts,
    how = "inner",
    predicate = "within",
)


In [ ]:
joined_gpd.shape

In [ ]:
joined_gpd.columns

In [ ]:
joined_gpd.head()

#### Count number of parties per tract

In [ ]:
pd.Series.reset_index

In [ ]:
incident_counts_df = joined_gpd["GEOID"].value_counts().reset_index( name = "party_count")
incident_counts_df

#### Merge tracts with party counts per tract

In [ ]:
tracts.merge(incident_counts_df, how="left", on="GEOID").sort_values( by = ["party_count"], ascending = False ).fillna("0")


Keep tracks that have had at least one incident

In [ ]:
tracts_abq = tracts[tracts['incidents'] != 0].copy()
tracts_abq.head()


In [ ]:
tracts_abq.set_index('TRACTCE', inplace = True)


In [ ]:
tracts_abq


In [ ]:
fig = px.choropleth_map(tracts_abq,
                            geojson = tracts_abq.geometry,
                            locations=tracts_abq.index,
                            color='incidents',
                            color_continuous_scale="Viridis",
                            map_style = 'carto-positron',
                            zoom = 10,
                            center = {"lat": 35.0844, "lon": -106.6504},
                            opacity = .7
                           )
fig.update_geos(fitbounds="locations", visible=False)
fig.show()
